# Attack MNIST Recognition

### Summary
- Overall it seems that CNN should be the best solution for image recognition, whilst Transformer (Encoder) tends to be the more comprehesive one(in terms of the overall relationship), downside is Transformer (Encoder) is not too sensitive to the orders.
- After 3 attacks are applied, the ASR overall for MLP are the highest ~99.96%, it shows that MLP tends to be the less robust model, when it comes to the most robust model, it's CNN, the ASR is ~76.49%, due to it looks at small local patches (like 3×3) and learns edges/strokes, and pooling makes it less sensitive to tiny changes.
- Among 3 attacks, the most likely to fool models is PGD/I-FGSM, as it consists of many small steps, each step recalculates the gradient, and intuitively more steps = stronger attack.

Attack 1: FGSM, eps=0.2
- MLP:
    - Recognition rate (before attack): 94.34%
    - ASR: 99.76%
- CNN
    - Recognition rate (before attack): 98.13%
    - ASR: 52.47%
- Encoder
    - Recognition rate (before attack): 96.02%
    - ASR: 87.00%

Attack 2: PGD/I-FGSM, eps=0.2, alpha=0.01, steps=20
- MLP:
    - Recognition rate (before attack): 94.34%
    - ASR: 99.96%
- CNN
    - Recognition rate (before attack): 98.13%
    - ASR: 73.52%
- Encoder
    - Recognition rate (before attack): 96.02%
    - ASR: 99.69%

Attack 3: MI-FGSM, eps=0.2, alpha=0.01, steps=20
- MLP:
    - Recognition rate (before attack): 94.34%
    - ASR: 99.92%
- CNN
    - Recognition rate (before attack): 98.13%
    - ASR: 76.49%
- Encoder
    - Recognition rate (before attack): 96.02%
    - ASR: 98.15%





In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
import torchvision.transforms as T

In [2]:
# load the dataset
mnist_dataset = load_dataset("ylecun/mnist")
train_data = mnist_dataset['train']
test_data = mnist_dataset['test']

Found cached dataset parquet (/Users/Lusa/.cache/huggingface/datasets/ylecun___parquet/mnist-7d7961f323d0d851/0.0.0/14a00e99c0d15a23649d0db8944380ac81082d4b021f398733dd84f3a6c569a7)


  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# example check - get the first image and label
to_tensor = T.ToTensor() 
example = train_data[0]
image = to_tensor(example['image'])
label = example['label']
print(image.shape)

torch.Size([1, 28, 28])


In [4]:
# set the format for pytorch
mnist_dataset.set_format(type='torch', columns=['image', 'label'])

In [5]:
mnist_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [6]:
# set the batch
train_batch = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_batch = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

## Define MLP, CNN, Transformers

In [7]:
#model functions

def mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

def cnn():
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(64 * 5 * 5, 10)
    )

def transformer(nhead=4, num_layers=6):
    encoder_layer = nn.TransformerEncoderLayer(d_model=28, nhead=nhead, batch_first=True)
    return nn.Sequential(
        nn.TransformerEncoder(encoder_layer, num_layers=num_layers),
        nn.Flatten(),
        nn.Linear(28 * 28, 10)
    )

#train function

def train_one_epoch(model, train_batch, optimizer, loss_fn, x_fn):
    model.train()
    for batch in train_batch:
        x = x_fn(batch)
        y = batch["label"]

        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return loss.item()

#test function
def test_accuracy(model, test_batch, x_fn):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_batch:
            x = x_fn(batch)
            y = batch["label"]

            outputs = model(x)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    return 100.0 * correct / total

# standard input for each model

x_mlp = lambda batch: batch["image"].float() / 255.0
x_cnn = lambda batch: batch["image"].float().unsqueeze(1) / 255.0   # add channel
x_trf = lambda batch: batch["image"].float() / 255.0         

loss_fn = nn.CrossEntropyLoss()

## Define - 
### 1) Fast Gradient Sign Method (FGSM) 
### 2) Iterative FGSM (I-FGSM) / Projected Gradient Descent (PGD) 
### 3)Momentum I-FGSM
## Calculate - 
### 1) recognition rate before attacks
### 2) attack success rate (ASR)

In [8]:
# Recognition rate before attacks = clean accuracy (%) on test set
def recognition_rate(model, test_batch, x_fn):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_batch:
            x = x_fn(batch)
            y = batch["label"]
            logits = model(x)
            pred = logits.argmax(dim=1)
            total += y.size(0)
            correct += (pred == y).sum().item()
    return 100.0 * correct / total


# FGSM
def fgsm_attack(model, x, y, loss_fn, eps):
    x_adv = x.detach().clone().requires_grad_(True)

    logits = model(x_adv)
    loss = loss_fn(logits, y)

    model.zero_grad()
    if x_adv.grad is not None:
        x_adv.grad.zero_()
    loss.backward()

    x_adv = x_adv + eps * x_adv.grad.sign()
    return x_adv.detach().clamp(0.0, 1.0)

# I-FGSM / PGD iterative FGSM + projection back to eps-ball around x
def pgd_attack(model, x, y, loss_fn, eps, alpha, steps, random_start=True):
    x_orig = x.detach()
    if random_start:
        x_adv = (x_orig + torch.empty_like(x_orig).uniform_(-eps, eps)).clamp(0.0, 1.0)
    else:
        x_adv = x_orig.clone()

    for _ in range(steps):
        x_adv = x_adv.detach().clone().requires_grad_(True)

        logits = model(x_adv)
        loss = loss_fn(logits, y)

        model.zero_grad()
        if x_adv.grad is not None:
            x_adv.grad.zero_()
        loss.backward()

        # step
        x_adv = x_adv + alpha * x_adv.grad.sign()

        delta = torch.clamp(x_adv - x_orig, min=-eps, max=eps)
        x_adv = (x_orig + delta).clamp(0.0, 1.0)

    return x_adv.detach()

# Momentum I-FGSM
def mifgsm_attack(model, x, y, loss_fn, eps, alpha, steps, decay=1.0):
    x_orig = x.detach()
    x_adv = x_orig.clone()
    g = torch.zeros_like(x_orig)

    for _ in range(steps):
        x_adv = x_adv.detach().clone().requires_grad_(True)

        logits = model(x_adv)
        loss = loss_fn(logits, y)

        model.zero_grad()
        if x_adv.grad is not None:
            x_adv.grad.zero_()
        loss.backward()

        grad = x_adv.grad.detach()

        grad_norm = grad / (grad.abs().mean(dim=list(range(1, grad.dim())), keepdim=True) + 1e-12)

        g = decay * g + grad_norm
        x_adv = x_adv + alpha * g.sign()

        delta = torch.clamp(x_adv - x_orig, min=-eps, max=eps)
        x_adv = (x_orig + delta).clamp(0.0, 1.0)

    return x_adv.detach()


# Attack Success Rate (ASR)
# among samples that the model classifies correctly on clean inputs,how many become misclassified after the attack.
def attack_success_rate(model, test_batch, x_fn, loss_fn, attack_fn, **attack_kwargs):
    model.eval()
    total_correct_clean = 0
    total_success = 0

    for batch in test_batch:
        x = x_fn(batch)
        y = batch["label"]

        # clean prediction
        with torch.no_grad():
            clean_pred = model(x).argmax(dim=1)

        mask = (clean_pred == y)
        if mask.sum().item() == 0:
            continue

        x_corr = x[mask]
        y_corr = y[mask]

        # adversarial examples (needs grad)
        x_adv = attack_fn(model, x_corr, y_corr, loss_fn=loss_fn, **attack_kwargs)

        with torch.no_grad():
            adv_pred = model(x_adv).argmax(dim=1)

        total_correct_clean += y_corr.size(0)
        total_success += (adv_pred != y_corr).sum().item()

    if total_correct_clean == 0:
        return 0.0
    return 100.0 * total_success / total_correct_clean


# apply to 3 models

def run_model(name, model, x_fn, train_batch, test_batch, lr=0.001):
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    final_loss = train_one_epoch(model, train_batch, opt, loss_fn, x_fn)
    print(f"{name} - Training Final Loss: {final_loss:.3f}")

    rr = recognition_rate(model, test_batch, x_fn)
    print(f"{name} - Recognition rate (clean): {rr:.2f}%")

    # ASR for each attack (test some defaults)
    eps = 0.2
    alpha = 0.01
    steps = 20

    asr_fgsm = attack_success_rate(model, test_batch, x_fn, loss_fn, fgsm_attack, eps=eps)
    print(f"{name} - ASR (FGSM, eps={eps}): {asr_fgsm:.2f}%")

    asr_pgd = attack_success_rate(model, test_batch, x_fn, loss_fn, pgd_attack,
                                  eps=eps, alpha=alpha, steps=steps, random_start=True)
    print(f"{name} - ASR (PGD/I-FGSM, eps={eps}, alpha={alpha}, steps={steps}): {asr_pgd:.2f}%")

    asr_mi = attack_success_rate(model, test_batch, x_fn, loss_fn, mifgsm_attack,
                                 eps=eps, alpha=alpha, steps=steps, decay=1.0)
    print(f"{name} - ASR (MI-FGSM, eps={eps}, alpha={alpha}, steps={steps}): {asr_mi:.2f}%")

    return model

# run all models
model_mlp = run_model("MLP", mlp(), x_mlp, train_batch, test_batch)
model_cnn = run_model("CNN", cnn(), x_cnn, train_batch, test_batch)
model_ecdr = run_model("Transformer", transformer(nhead=4, num_layers=6), x_trf, train_batch, test_batch)

MLP - Training Final Loss: 0.187
MLP - Recognition rate (clean): 94.34%
MLP - ASR (FGSM, eps=0.2): 99.76%
MLP - ASR (PGD/I-FGSM, eps=0.2, alpha=0.01, steps=20): 99.96%
MLP - ASR (MI-FGSM, eps=0.2, alpha=0.01, steps=20): 99.92%
CNN - Training Final Loss: 0.048
CNN - Recognition rate (clean): 98.13%
CNN - ASR (FGSM, eps=0.2): 52.47%
CNN - ASR (PGD/I-FGSM, eps=0.2, alpha=0.01, steps=20): 73.52%
CNN - ASR (MI-FGSM, eps=0.2, alpha=0.01, steps=20): 76.49%
Transformer - Training Final Loss: 0.079
Transformer - Recognition rate (clean): 96.02%
Transformer - ASR (FGSM, eps=0.2): 87.00%
Transformer - ASR (PGD/I-FGSM, eps=0.2, alpha=0.01, steps=20): 99.69%
Transformer - ASR (MI-FGSM, eps=0.2, alpha=0.01, steps=20): 98.15%
